# Stable Diffusion: Text-to-Image Generation in Python

**Assignment:** Implement a text-to-image generation model (Stable Diffusion).

**Reference:** [GeeksforGeeks tutorial](https://www.geeksforgeeks.org/deep-learning/generate-images-from-text-in-python-stable-diffusion/)

This notebook walks through every step of generating an image from a text prompt using the `CompVis/stable-diffusion-v1-4` model and the Hugging Face `diffusers` library.

> **Tip:** In Google Colab, set the runtime to **T4 GPU** (Runtime → Change runtime type → T4 GPU) before running the cells.

## Step 1 — Install the required libraries

- **diffusers**: the main Hugging Face package that wraps the entire Stable Diffusion pipeline (text encoder + U-Net + VAE + scheduler).
- **transformers**: provides the CLIP text encoder used to turn the prompt into embeddings.
- **accelerate**: lets the model use the GPU efficiently.
- **safetensors**: a fast, secure format for the model weights.
- **scipy**: numerical helpers used internally by some schedulers.
- **Pillow (PIL)**: image library used to display and save the final image.

In [ ]:
!pip install --quiet diffusers transformers accelerate safetensors scipy Pillow

## Step 2 — Import the libraries

- `torch` is the deep-learning framework that runs the neural network on the GPU.
- `StableDiffusionPipeline` is the high-level class that hides the diffusion math behind a simple call.
- `Image` from PIL represents the generated picture in memory.
- `display` lets us render the image inline in the notebook.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
from IPython.display import display

## Step 3 — Detect the device (GPU vs CPU)

Stable Diffusion is far too slow on a CPU for practical use, so we check whether CUDA (an NVIDIA GPU) is available. We also use `float16` (half-precision) on the GPU, which roughly halves the memory footprint and speeds inference up.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device}  (dtype={dtype})")

## Step 4 — Load the Stable Diffusion pipeline

`from_pretrained` downloads the model weights from the Hugging Face Hub the first time it runs (~4 GB) and caches them. We then move every sub-module of the pipeline (CLIP text encoder, U-Net, VAE) onto the GPU.

We pick **`CompVis/stable-diffusion-v1-4`** — the original Stable Diffusion model. The GeeksforGeeks tutorial recommends it as the lightweight option, and unlike the newer `stabilityai` repos it is publicly accessible without a Hugging Face token.

In [ ]:
MODEL_ID = "CompVis/stable-diffusion-v1-4"

pipeline = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)
pipeline = pipeline.to(device)

if device == "cuda":
    pipeline.enable_attention_slicing()  # fits the model on small GPUs

print("Pipeline loaded successfully.")

## Step 5 — Define the text prompt

The prompt is the most important input. Good prompts describe the **subject**, **setting**, **style**, and any extra **details** (lighting, mood, resolution, etc.).

A `negative_prompt` lists things we do *not* want in the image — a quick way to avoid common artefacts.

In [ ]:
prompt = (
    "A majestic dragon perched on a snowy mountain peak at sunset, "
    "breathtaking fantasy art, highly detailed scales, cinematic "
    "lighting, golden hour, ultra realistic, 8k"
)

negative_prompt = "blurry, low quality, distorted, deformed, watermark, text"

print("Prompt:", prompt)

## Step 6 — Generate the image

Calling the pipeline runs the full denoising loop:

1. The CLIP text encoder turns the prompt into 768-dim embeddings.
2. A random noise tensor is sampled in latent space.
3. The U-Net denoises this latent for `num_inference_steps` iterations, guided by the text embeddings.
4. The VAE decoder turns the cleaned latent into a full-resolution RGB image.

Key parameters:
- `num_inference_steps=50` — 50 denoising steps (the standard sweet spot between speed and quality).
- `guidance_scale=7.5` — how strictly the model follows the prompt.
- `height=512, width=512` — SD 1.x's native training resolution (must be multiples of 8).
- `generator` with `manual_seed(42)` — makes the result reproducible.

In [ ]:
generator = torch.Generator(device=device).manual_seed(42)

result = pipeline(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
    height=512,
    width=512,
    generator=generator,
)

image: Image.Image = result.images[0]
print("Image generated.")

## Step 7 — Display and save the image

`display(image)` renders the picture inline in the notebook. We also save it to `generated_image.png` so we can use it elsewhere.

In [ ]:
display(image)
image.save("generated_image.png")
print("Saved as generated_image.png")